# Motion Energy Distribution Analysis

This notebook analyzes the distribution of motion energy ($E_{acc}$, $E_{gyr}$) over the 1.5-second recording window across the collected training datasets. 

### Objectives:
1. **Global Analysis:** Aggregate all gesture records to see overall framing and stillness quality.
2. **Per-Gesture Analysis:** Inspect individual gesture curves to pinpoint dynamic patterns.
3. **Quality Ranking:** Calculate quantitative metrics (centering error, boundary stillness) to identify which gestures have the highest quality and which ones represent the worst training data (leakage or misalignment).

### Why?

We established this analysis of motion energy to work out how big recording windows etc. need to be. So we used it to  analyze  recording quality with the goal of evaluating and optimizing the data recording process.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Setup plotting backend
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Configuration
DATA_DIR = Path("../data")
SAMPLING_RATE_HZ = 100
WINDOW_SIZE_S = 1.5
TARGET_LENGTH = int(SAMPLING_RATE_HZ * WINDOW_SIZE_S)  # 150 samples

print(f"Target folder: {DATA_DIR.resolve()}")
print(f"Target dimensions: {TARGET_LENGTH} samples at {SAMPLING_RATE_HZ} Hz")

## Mathematical Definition of Motion Energy

For each time step $t$ in a gesture file, the motion energy is defined as the magnitude of the sensor vectors:

- **Accelerometer Motion Energy** ($E_{acc}$):
  $$E_{acc}(t) = \sqrt{a_x(t)^2 + a_y(t)^2 + a_z(t)^2}$$

- **Gyroscope Motion Energy (Rotational Energy)** ($E_{gyr}$):
  $$E_{gyr}(t) = \sqrt{\omega_x(t)^2 + \omega_y(t)^2 + \omega_z(t)^2}$$

We will compute these magnitudes for both **IMU1 (Finger)** and **IMU2 (Wrist)**.

In [ ]:
def calculate_sample_energy(csv_path: Path):
    """Loads a recording CSV and returns energy profiles normalized to target length."""
    df = pd.read_csv(csv_path)
    
    # Respect the start index .txt file if it exists
    txt_path = csv_path.with_suffix('.txt')
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            start_idx = int(f.read().strip())
        df = df.iloc[start_idx : start_idx + TARGET_LENGTH]
    else: 
        # Standardize length to exactly 150 samples
        if len(df) < TARGET_LENGTH:
            pad_size = TARGET_LENGTH - len(df)
            last_row = df.iloc[-1:]
            df = pd.concat([df, pd.concat([last_row] * pad_size, ignore_index=True)], ignore_index=True)
        elif len(df) > TARGET_LENGTH:
            df = df.iloc[:TARGET_LENGTH]
        
    profiles = {}
    # IMU1 (finger)
    profiles['IMU1_E_acc'] = np.sqrt(df['IMU1_accX']**2 + df['IMU1_accY']**2 + df['IMU1_accZ']**2)
    profiles['IMU1_E_gyr'] = np.sqrt(df['IMU1_gyrX']**2 + df['IMU1_gyrY']**2 + df['IMU1_gyrZ']**2)
    
    # IMU2 (wrist)
    profiles['IMU2_E_acc'] = np.sqrt(df['IMU2_accX']**2 + df['IMU2_accY']**2 + df['IMU2_accZ']**2)
    profiles['IMU2_E_gyr'] = np.sqrt(df['IMU2_gyrX']**2 + df['IMU2_gyrY']**2 + df['IMU2_gyrZ']**2)
    
    return profiles


## Processing Gesture Datasets

In [ ]:
gestures = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
print(f"Detected gesture categories: {gestures}")

gesture_profiles = {}
all_imu1_acc = []
all_imu1_gyr = []
all_imu2_acc = []
all_imu2_gyr = []

for gesture in gestures:
    # Find all CSV files recursively, excluding calibration.csv and energy_distribution.csv
    files = [Path(f) for f in DATA_DIR.glob(f"{gesture}/**/*.csv") if not Path(f).name.startswith("calibration") and not Path(f).name.startswith("energy_distribution")]
    if not files:
        print(f"  -> '{gesture}' is empty or contains no valid sample files.")
        continue
        
    print(f"Processing '{gesture}' ({len(files)} files)...")
    
    run_data = {
        'IMU1_acc': [], 'IMU1_gyr': [],
        'IMU2_acc': [], 'IMU2_gyr': []
    }
    
    for f in files:
        try:
            p = calculate_sample_energy(f)
            run_data['IMU1_acc'].append(p['IMU1_E_acc'])
            run_data['IMU1_gyr'].append(p['IMU1_E_gyr'])
            run_data['IMU2_acc'].append(p['IMU2_E_acc'])
            run_data['IMU2_gyr'].append(p['IMU2_E_gyr'])
            
            # Add to global aggregates
            all_imu1_acc.append(p['IMU1_E_acc'])
            all_imu1_gyr.append(p['IMU1_E_gyr'])
            all_imu2_acc.append(p['IMU2_E_acc'])
            all_imu2_gyr.append(p['IMU2_E_gyr'])
        except Exception as e:
            print(f"    Error parsing {f.name}: {e}")
            
    if run_data['IMU1_acc']:
        gesture_profiles[gesture] = {
            'IMU1_acc_runs': np.array(run_data['IMU1_acc']),
            'IMU1_gyr_runs': np.array(run_data['IMU1_gyr']),
            'IMU2_acc_runs': np.array(run_data['IMU2_acc']),
            'IMU2_gyr_runs': np.array(run_data['IMU2_gyr']),
            'count': len(run_data['IMU1_acc'])
        }
        print(f"    Successfully aggregated {gesture_profiles[gesture]['count']} samples.")

## 1. Global Analysis

We analyze the global motion energy profiles averaged across all gesture categories in the dataset.

In [ ]:
if all_imu1_acc:
    all_imu1_acc = np.array(all_imu1_acc)
    all_imu1_gyr = np.array(all_imu1_gyr)
    all_imu2_acc = np.array(all_imu2_acc)
    all_imu2_gyr = np.array(all_imu2_gyr)
    
    t = np.linspace(0, WINDOW_SIZE_S, TARGET_LENGTH)
    fig, axs = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
    fig.suptitle(f"Overall Motion Energy Distribution (Entire Dataset: {len(all_imu1_acc)} Samples)", fontsize=16, fontweight='bold')
    
    # --- IMU1 Accelerometer (Wrist) ---
    mean_acc1 = np.mean(all_imu1_acc, axis=0)
    std_acc1 = np.std(all_imu1_acc, axis=0)
    axs[0, 0].plot(t, mean_acc1, color='#1f77b4', label='Mean')
    axs[0, 0].fill_between(t, mean_acc1 - std_acc1, mean_acc1 + std_acc1, color='#1f77b4', alpha=0.15)
    axs[0, 0].set_title("IMU1 (Finger) Accelerometer Magnitude")
    axs[0, 0].set_ylabel("Acceleration (g)")
    axs[0, 0].grid(True, linestyle='--', alpha=0.7)
    axs[0, 0].legend(loc='upper right')
    
    # --- IMU1 Gyroscope (Wrist) ---
    mean_gyr1 = np.mean(all_imu1_gyr, axis=0)
    std_gyr1 = np.std(all_imu1_gyr, axis=0)
    axs[1, 0].plot(t, mean_gyr1, color='#ff7f0e', label='Mean')
    axs[1, 0].fill_between(t, mean_gyr1 - std_gyr1, mean_gyr1 + std_gyr1, color='#ff7f0e', alpha=0.15)
    axs[1, 0].set_title("IMU1 (Finger) Gyroscope Magnitude")
    axs[1, 0].set_ylabel("Angular Velocity (dps)")
    axs[1, 0].set_xlabel("Time (seconds)")
    axs[1, 0].grid(True, linestyle='--', alpha=0.7)
    axs[1, 0].legend(loc='upper right')
    
    # --- IMU2 Accelerometer (Finger) ---
    mean_acc2 = np.mean(all_imu2_acc, axis=0)
    std_acc2 = np.std(all_imu2_acc, axis=0)
    axs[0, 1].plot(t, mean_acc2, color='#2ca02c', label='Mean')
    axs[0, 1].fill_between(t, mean_acc2 - std_acc2, mean_acc2 + std_acc2, color='#2ca02c', alpha=0.15)
    axs[0, 1].set_title("IMU2 (Wrist) Accelerometer Magnitude")
    axs[0, 1].grid(True, linestyle='--', alpha=0.7)
    axs[0, 1].legend(loc='upper right')
    
    # --- IMU2 Gyroscope (Finger) ---
    mean_gyr2 = np.mean(all_imu2_gyr, axis=0)
    std_gyr2 = np.std(all_imu2_gyr, axis=0)
    axs[1, 1].plot(t, mean_gyr2, color='#d62728', label='Mean')
    axs[1, 1].fill_between(t, mean_gyr2 - std_gyr2, mean_gyr2 + std_gyr2, color='#d62728', alpha=0.15)
    axs[1, 1].set_title("IMU2 (Wrist) Gyroscope Magnitude")
    axs[1, 1].set_xlabel("Time (seconds)")
    axs[1, 1].grid(True, linestyle='--', alpha=0.7)
    axs[1, 1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
else:
    print("No aggregated runs to analyze globally.")

## 2. Per-Gesture Analysis

In [ ]:
t = np.linspace(0, WINDOW_SIZE_S, TARGET_LENGTH)

for gesture, data in gesture_profiles.items():
    fig, axs = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
    fig.suptitle(f"Motion Energy Profiles: Gesture '{gesture}' ({data['count']} Samples)", fontsize=16, fontweight='bold')
    
    # --- IMU1 Accelerometer ---
    mean_acc1 = np.mean(data['IMU1_acc_runs'], axis=0)
    std_acc1 = np.std(data['IMU1_acc_runs'], axis=0)
    axs[0, 0].plot(t, mean_acc1, color='#1f77b4', label='Mean')
    axs[0, 0].fill_between(t, mean_acc1 - std_acc1, mean_acc1 + std_acc1, color='#1f77b4', alpha=0.15)
    axs[0, 0].set_title("IMU1 (Finger) Accelerometer Magnitude")
    axs[0, 0].set_ylabel("Acceleration (g)")
    axs[0, 0].grid(True, linestyle='--', alpha=0.7)
    axs[0, 0].legend(loc='upper right')
    
    # --- IMU1 Gyroscope ---
    mean_gyr1 = np.mean(data['IMU1_gyr_runs'], axis=0)
    std_gyr1 = np.std(data['IMU1_gyr_runs'], axis=0)
    axs[1, 0].plot(t, mean_gyr1, color='#ff7f0e', label='Mean')
    axs[1, 0].fill_between(t, mean_gyr1 - std_gyr1, mean_gyr1 + std_gyr1, color='#ff7f0e', alpha=0.15)
    axs[1, 0].set_title("IMU1 (Finger) Gyroscope Magnitude")
    axs[1, 0].set_ylabel("Angular Velocity (dps)")
    axs[1, 0].set_xlabel("Time (seconds)")
    axs[1, 0].grid(True, linestyle='--', alpha=0.7)
    axs[1, 0].legend(loc='upper right')
    
    # --- IMU2 Accelerometer ---
    mean_acc2 = np.mean(data['IMU2_acc_runs'], axis=0)
    std_acc2 = np.std(data['IMU2_acc_runs'], axis=0)
    axs[0, 1].plot(t, mean_acc2, color='#2ca02c', label='Mean')
    axs[0, 1].fill_between(t, mean_acc2 - std_acc2, mean_acc2 + std_acc2, color='#2ca02c', alpha=0.15)
    axs[0, 1].set_title("IMU2 (Wrist) Accelerometer Magnitude")
    axs[0, 1].grid(True, linestyle='--', alpha=0.7)
    axs[0, 1].legend(loc='upper right')
    
    # --- IMU2 Gyroscope ---
    mean_gyr2 = np.mean(data['IMU2_gyr_runs'], axis=0)
    std_gyr2 = np.std(data['IMU2_gyr_runs'], axis=0)
    axs[1, 1].plot(t, mean_gyr2, color='#d62728', label='Mean')
    axs[1, 1].fill_between(t, mean_gyr2 - std_gyr2, mean_gyr2 + std_gyr2, color='#d62728', alpha=0.15)
    axs[1, 1].set_title("IMU2 (Wrist) Gyroscope Magnitude")
    axs[1, 1].set_xlabel("Time (seconds)")
    axs[1, 1].grid(True, linestyle='--', alpha=0.7)
    axs[1, 1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

## 3. Gesture Data Quality Score

To analyze where the training data quality is highest and which gestures represent the worst/messiest training data, we define three mathematical quality metrics:

1. **Start Stillness (dps):** Mean wrist gyroscope magnitude ($E_{gyr}$) in the first 20 samples (0.0 to 0.2s). Lower is better ($<8$ dps is excellent).
2. **End Stillness (dps):** Mean wrist gyroscope magnitude ($E_{gyr}$) in the last 20 samples (1.3 to 1.5s). Lower is better ($<8$ dps is excellent).
3. **Centering Error (seconds):** The absolute offset of the peak rotation energy from the exact center (0.75s). Lower is better ($<0.15$s is excellent).

Gestures with high boundary stillness values suffer from **motion leakage** (i.e. starting the gesture too early or moving arm after ending). Gestures with high centering error suffer from **unaligned peak windows**.

In [ ]:
quality_metrics = []

for gesture, data in gesture_profiles.items():
    # config/devices.yml: IMU1 is finger (index), IMU2 is wrist
    # We audit arm stillness using the wrist sensor (IMU2)
    mean_gyr_wrist = np.mean(data['IMU2_gyr_runs'], axis=0)
    mean_gyr_finger = np.mean(data['IMU1_gyr_runs'], axis=0)
    
    # Stillness metric on the wrist sensor (first and last 20 samples)
    start_stillness = np.mean(mean_gyr_wrist[:20])
    end_stillness = np.mean(mean_gyr_wrist[-20:])
    
    # Peak centering on the finger sensor (primary movement indicator)
    peak_idx = np.argmax(mean_gyr_finger)
    peak_time = peak_idx * (WINDOW_SIZE_S / TARGET_LENGTH)
    centering_error = abs(peak_time - 0.75)
    
    # Additional Data Recording Quality Metrics:
    # 1. Signal-to-Noise Ratio (SNR) of finger gyroscope signal
    peak_signal = np.max(mean_gyr_finger)
    noise_floor = np.mean(mean_gyr_finger[:20]) + 1e-5
    snr_db = 20 * np.log10(peak_signal / noise_floor)
    
    # 2. Co-activation Index (Wrist vs. Finger motion energy deviation)
    finger_motion = np.abs(data['IMU1_acc_runs'] - 1.0) + 0.01 * data['IMU1_gyr_runs']
    wrist_motion = np.abs(data['IMU2_acc_runs'] - 1.0) + 0.01 * data['IMU2_gyr_runs']
    mean_wrist = np.mean(wrist_motion)
    mean_finger = np.mean(finger_motion)
    co_activation = mean_wrist / (mean_finger + 1e-5)
    
    # Classify Quality based on wrist stillness and centering error
    if start_stillness < 8.0 and end_stillness < 8.0 and centering_error < 0.12:
        rating = "🟢 Excellent"
    elif start_stillness < 15.0 and end_stillness < 15.0 and centering_error < 0.22:
        rating = "🟡 Good"
    else:
        rating = "🔴 Needs Improvement"
        
    quality_metrics.append({
        'Gesture': gesture,
        'Number of Samples': data['count'],
        'Start Stillness (dps)': round(start_stillness, 2),
        'End Stillness (dps)': round(end_stillness, 2),
        'Peak Time (s)': round(peak_time, 2),
        'Centering Error (s)': round(centering_error, 2),
        'SNR (dB)': round(snr_db, 1),
        'Co-activation': round(co_activation, 2),
        'Quality Rating': rating
    })

df_quality = pd.DataFrame(quality_metrics)
# Rank gestures: Excellent / Good / Needs Improvement based on a composite penalty
if not df_quality.empty:
    df_quality['penalty'] = df_quality['Centering Error (s)'] * 50 + df_quality['Start Stillness (dps)'] + df_quality['End Stillness (dps)']
    df_quality = df_quality.sort_values(by='penalty').drop(columns=['penalty'])

print("=========================================================================")
print("            Gesture Training Data Quality Score Comparison               ")
print("=========================================================================")
display(df_quality)


## 3. Window Slicing Comparison (Alignment Evaluation)

We compare the total motion energy contained in three different slicing strategies over the 1.7-second raw recording window:
1. **First 150 samples:** `df.iloc[0:150]`
2. **Last 150 samples:** `df.iloc[-150:]`
3. **Centered 150 samples:** `df.iloc[start_idx : start_idx + 150]` (using the `.txt` start index)

This comparison verifies whether the centroid-centering algorithm successfully captures the peak energy segment of the gesture compared to blind edge-slicing.

In [ ]:
# Slicing comparison analysis
comparison_records = []

for gesture in gestures:
    files = [Path(f) for f in DATA_DIR.glob(f"{gesture}/**/*.csv") if not Path(f).name.startswith("calibration") and not Path(f).name.startswith("energy_distribution")]
    
    for f in files:
        df = pd.read_csv(f)
        txt_path = f.with_suffix('.txt')
        if not txt_path.exists() or len(df) < TARGET_LENGTH:
            continue
            
        with open(txt_path, "r", encoding="utf-8") as tf:
            start_idx = int(tf.read().strip())
            
        # Calculate motion energy
        def get_energy_sum(w_df):
            a1 = np.sqrt(w_df['IMU1_accX']**2 + w_df['IMU1_accY']**2 + w_df['IMU1_accZ']**2) if 'IMU1_accX' in w_df.columns else np.zeros(len(w_df))
            a2 = np.sqrt(w_df['IMU2_accX']**2 + w_df['IMU2_accY']**2 + w_df['IMU2_accZ']**2) if 'IMU2_accX' in w_df.columns else np.zeros(len(w_df))
            g1 = np.sqrt(w_df['IMU1_gyrX']**2 + w_df['IMU1_gyrY']**2 + w_df['IMU1_gyrZ']**2) if 'IMU1_gyrX' in w_df.columns else np.zeros(len(w_df))
            g2 = np.sqrt(w_df['IMU2_gyrX']**2 + w_df['IMU2_gyrY']**2 + w_df['IMU2_gyrZ']**2) if 'IMU2_gyrX' in w_df.columns else np.zeros(len(w_df))
            e = np.abs(a1 - 1.0) + np.abs(a2 - 1.0) + 0.01 * (g1 + g2)
            return float(e.sum())
            
        e_first = get_energy_sum(df.iloc[:TARGET_LENGTH])
        e_last = get_energy_sum(df.iloc[-TARGET_LENGTH:])
        e_centered = get_energy_sum(df.iloc[start_idx : start_idx + TARGET_LENGTH])
        
        comparison_records.append({
            'Gesture': gesture,
            'File': f.name,
            'First 150': e_first,
            'Last 150': e_last,
            'Centered 150': e_centered
        })

df_comp = pd.DataFrame(comparison_records)
if not df_comp.empty:
    summary_comp = df_comp.groupby('Gesture')[['First 150', 'Last 150', 'Centered 150']].mean().reset_index()
    print("=========================================================================")
    print("        Average Motion Energy Comparison per Window Slicing              ")
    print("=========================================================================")
    display(summary_comp)
    
    # Plotting comparison
    summary_comp.plot(x='Gesture', y=['First 150', 'Last 150', 'Centered 150'], kind='bar', figsize=(10, 5))
    plt.title('Motion Energy Comparison across Slicing Strategies')
    plt.ylabel('Average Window Motion Energy')
    plt.xlabel('Gesture Category')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No sample files with .txt companion files found for slicing comparison.")

### 3.1 Slicing Distribution Overlays (Gesture Alignment Visual Impact)

We overlay the motion energy curves of the three slicing strategies to visually demonstrate the translation effects. Peaks are marked on each distribution, with axes scale standardized for visual comparison.

In [ ]:
# Overlay distribution comparison plot for each gesture
for gesture in gestures:
    files = [Path(f) for f in DATA_DIR.glob(f"{gesture}/**/*.csv") if not Path(f).name.startswith("calibration") and not Path(f).name.startswith("energy_distribution")]
    
    first_runs = []
    last_runs = []
    centered_runs = []
    
    for f in files:
        df = pd.read_csv(f)
        txt_path = f.with_suffix('.txt')
        if not txt_path.exists() or len(df) < TARGET_LENGTH:
            continue
            
        with open(txt_path, "r", encoding="utf-8") as tf:
            start_idx = int(tf.read().strip())
            
        # Calculate full energy
        a1 = np.sqrt(df['IMU1_accX']**2 + df['IMU1_accY']**2 + df['IMU1_accZ']**2) if 'IMU1_accX' in df.columns else np.zeros(len(df))
        a2 = np.sqrt(df['IMU2_accX']**2 + df['IMU2_accY']**2 + df['IMU2_accZ']**2) if 'IMU2_accX' in df.columns else np.zeros(len(df))
        g1 = np.sqrt(df['IMU1_gyrX']**2 + df['IMU1_gyrY']**2 + df['IMU1_gyrZ']**2) if 'IMU1_gyrX' in df.columns else np.zeros(len(df))
        g2 = np.sqrt(df['IMU2_gyrX']**2 + df['IMU2_gyrY']**2 + df['IMU2_gyrZ']**2) if 'IMU2_gyrX' in df.columns else np.zeros(len(df))
        energy = np.abs(a1 - 1.0) + np.abs(a2 - 1.0) + 0.01 * (g1 + g2)
        
        first_runs.append(energy.iloc[:TARGET_LENGTH].values)
        last_runs.append(energy.iloc[-TARGET_LENGTH:].values)
        centered_runs.append(energy.iloc[start_idx : start_idx + TARGET_LENGTH].values)
        
    if not centered_runs:
        continue
        
    mean_first = np.mean(first_runs, axis=0)
    mean_last = np.mean(last_runs, axis=0)
    mean_centered = np.mean(centered_runs, axis=0)
    
    x_vals = range(TARGET_LENGTH)
    
    plt.figure(figsize=(10, 5))
    plt.plot(x_vals, mean_first, label='First 150 (Edge)', color='orange', alpha=0.8)
    plt.plot(x_vals, mean_last, label='Last 150 (Edge)', color='purple', alpha=0.8)
    plt.plot(x_vals, mean_centered, label='Centered 150 (Companion)', color='green', linewidth=2)
    
    # Find peaks
    peak_first_idx = np.argmax(mean_first)
    plt.scatter(peak_first_idx, mean_first[peak_first_idx], color='darkorange', zorder=5, s=70, edgecolors='black')
    plt.text(peak_first_idx + 2, mean_first[peak_first_idx], f"{mean_first[peak_first_idx]:.2f}", color='darkorange', fontsize=10, fontweight='bold')
    
    peak_last_idx = np.argmax(mean_last)
    plt.scatter(peak_last_idx, mean_last[peak_last_idx], color='indigo', zorder=5, s=70, edgecolors='black')
    plt.text(peak_last_idx + 2, mean_last[peak_last_idx], f"{mean_last[peak_last_idx]:.2f}", color='indigo', fontsize=10, fontweight='bold')
    
    peak_centered_idx = np.argmax(mean_centered)
    plt.scatter(peak_centered_idx, mean_centered[peak_centered_idx], color='darkgreen', zorder=5, s=70, edgecolors='black')
    plt.text(peak_centered_idx + 2, mean_centered[peak_centered_idx], f"{mean_centered[peak_centered_idx]:.2f}", color='darkgreen', fontsize=10, fontweight='bold')
    
    plt.title(f"Overlay Motion Energy Slicing: '{gesture}'")
    plt.xlabel("Sample Index in 150-sample Window")
    plt.ylabel("Average Motion Energy")
    plt.xlim(-5, TARGET_LENGTH + 5)
    
    # Keep similar ranges for x-axis and y-axis
    max_y = max(np.max(mean_first), np.max(mean_last), np.max(mean_centered))
    plt.ylim(-0.05, max_y * 1.25)
    
    plt.legend()
    plt.grid(True, linestyle=':')
    plt.tight_layout()
    plt.show()

## 4. Interpretation and Diagnosis

Review the ranking table above to diagnose dataset improvements:

- **High-Quality Gestures (🟢 Excellent):** The boundaries are stationary (gyroscope $<8$ dps), and the peak is centered in the window. These samples will yield excellent CNN classifier representation.
- **Poor-Quality Gestures (🔴 Needs Improvement):**
  - If **Start/End Stillness** is high: The user is moving their arm during the designated stillness phases. *Fix: Hold the wrist/hand completely still for 0.2s before initiating the gesture and immediately after finishing it.*
  - If **Centering Error** is high: The user is rushing the gesture (resulting in a left-shifted peak) or delaying it (right-shifted peak). *Fix: Attempt to perform the gesture peak speed exactly in the middle (0.75s) of the 1.5-second recording phase.*